In [ ]:
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import trange

scope = cw.scope()
target = cw.target(scope, cw.targets.SimpleSerial)
scope.default_setup()
scope.adc.samples=1  # what is this?

Add here your docs

In [ ]:
%%bash
make PLATFORM=CWLITEARM SOURCE=chipwhisperer_hw2_dpa.c

Add here your docs

In [ ]:
hex = 'output-CWLITEARM.hex'
cw.program_target(scope, cw.programmers.STM32FProgrammer, hex)

In [ ]:
import numpy as np

class CWHandler:
    def __init__(self, adc_samples_num: int):
        """
        Initializes a CWHandler object
        Args:
            adc_samples_num (int): Number of sampmles within a 
            trace outputed by the Chip Whisperer
        """
        self.scope = cw.scope()
        self.target = cw.target(scope, cw.targets.SimpleSerial)
        self.scope.default_target()
        self.scope.adc.samples = adc_samples_num

    def program_target(self, hex_file: str):
        """
        Programs the STM32 target on the Chip Whisperer
        Args:
            hex_file (str): hex_file
        """
        cw.program_target(self.scope, cw.programmers.STM32FProgrammer, hex_file)
        
    def send_command(self, opcode: str, payload: bytes):
        self.target.flush()
        self.target.simpleserial_write(opcode, payload)
        ret = scope.capture()
        if ret:
            raise Exception("Error in capturing response from command")
        return ret

    def send_command_capture_trace(self, opcode: str, payload: bytes) -> np.ndarray:
        self.target.flush()
        self.scope.arm()
        
        self.target.simpleserial_write(opcode, payload)
        
        ret = scope.capture()
        if ret:
            raise Exception("Error in capturing response from command")
        return ret

        
        
        

Adding execute trace as a function

In [ ]:
def execute_trace(command: str, payload: bytes):
    """
    Executes a trace using the given command and input string, with the provided scope and target objects.

    Parameters:
        command (str): The command to send to the target.
        payload (bytes): The input data to send to the target.
        scope: The scope object for controlling the capture process.
        target: The target object for sending and receiving data.

    Returns:
        tuple: A tuple containing the capture trace and the response from the target.
    """
    try:
        # Flush target and arm the scope
        target.flush()
        scope.arm()

        # Convert input to bytearray
        input_data = payload

        # Send command and data to target
        target.simpleserial_write(command, input_data)

        # Capture the trace
        ret = scope.capture()
        if ret:
            print("ERROR: Capture failed.")
            return None, None

        # Retrieve the trace and the target response
        trace = scope.get_last_trace()
        response = target.simpleserial_read('r', 1).decode('utf-8')

        return trace, response
    except Exception as e:
        print(f"An error occurred: {e}")
        return None, None

Now let us try to execute a simple encryption:

In [ ]:
from matplotlib import pyplot as plt

trace, resp = execute_trace('p', bytes(8))  # Inputting 16 bytes of 0 as an input to the AES encrypting

print(f"Encryption result is: {resp}")

print(type(trace))

Add here your docs

In [ ]:
scope.dis()
target.dis()